In [ ]:
PROJECT_NAME = '{{cookiecutter.project_name}}'
ENVIRON = 'local'

In [ ]:
# run the following to detect environment and run appropriate set-up scripts

import os

def test_for_colab() -> bool:
    try:
        from google.colab import drive  # type: ignore
        print("Running in Google Colab environment. Mounting Google Drive...")
        drive.mount('/content/drive')
        return True
    except ImportError:
        return False

if test_for_colab():
    import sys
    from pathlib import Path
    #sys.path.append(str(Path.cwd()))
    ENVIRON = 'colab'
    print("Running in Google Colab environment. Now setting github credentials needed to clone")
    !mkdir -p ~/.ssh/
    !cp /content/drive/MyDrive/colabSSH/id_rsa ~/.ssh/id_rsa
    !cp /content/drive/MyDrive/colabSSH/id_rsa.pub ~/.ssh/id_rsa.pub
    !cp /content/drive/MyDrive/colabSSH/known_hosts ~/.ssh/known_hosts
    !chmod 600 ~/.ssh/id_rsa
    !pwd
    !git clone git@github.com:essans/{PROJECT_NAME}.git
    %cd {PROJECT_NAME}
    print('--> installing required libraries')
    !pip install -e . --no-deps
    !pip install boto3
    sys.path.append(f'{str(Path().resolve())}/src')

    from src.utils.colab import (
        colab_upload_data_and_setup_path,
        colab_set_hf_token
        )
    
    # un-comment if files need to be uploaded to g-drive
    # colab_upload_data_and_setup_path(PROJECT_NAME) 

elif os.getenv('USER') in ['ubuntu', 'ec2-user']:
    ENVIRON = 'ec2'
    from utils import aws_utils
    print("Running in AWS EC2 environment.")

else:
    ENVIRON = 'local'
    print("Running in local environment.")v

In [ ]:
import os, sys
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np

from utils import helper

from utils.logging_utils import setup_logging
setup_logging(log_level=logging.INFO, log_to_screen=True)

In [ ]:
try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    ENVIRON = 'colab'
    

In [ ]:
project_dir = helper.get_project_root()
configs = helper.configs_from_yaml('configs/settings.yaml')
data_dir = configs.input_data.raw_dir
processed_dir = configs.input_data.processed_dir
outputs_dir = configs.outputs.outputs_dir

helper.set_hf_creds(ENVIRON)
   
helper.pd_format()

---